In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.features import calcular_surpresa, calcular_ian, calcular_ice

eventos_cpi = pd.read_csv("../data/eventos.csv")
print(eventos_cpi.head())

eventos_cpi = calcular_surpresa(eventos_cpi)
print(eventos_cpi[["data", "actual", "forecast", "surpresa_zscore"]].head())

  indicador        data  actual  forecast  diferenca  surpresa_zscore
0   CPI_EUA  2026-07-14    -0.4      -0.1       -0.3        -3.022518
1   CPI_EUA  2026-06-10     0.5       0.5        0.0         0.000000
2   CPI_EUA  2026-05-12     0.6       0.6        0.0         0.000000
3   CPI_EUA  2026-04-10     0.9       1.0       -0.1        -1.007506
4   CPI_EUA  2026-03-11     0.3       0.3        0.0         0.000000
         data  actual  forecast  surpresa_zscore
0  2026-07-14    -0.4      -0.1        -0.005917
1  2026-06-10     0.5       0.5         0.000000
2  2026-05-12     0.6       0.6         0.000000
3  2026-04-10     0.9       1.0        -0.001972
4  2026-03-11     0.3       0.3         0.000000


In [16]:
eventos_cpi_only = calcular_surpresa(eventos_cpi_only)
eventos_cpi_only = calcular_ian(eventos_cpi_only, termos_busca=["CPI", "inflation report"], geo="US")
eventos_cpi_only = calcular_ice(
    eventos_cpi_only,
    termos_otimistas=["abrir empresa", "comprar carro", "promoção passagens"],
    termos_pessimistas=["perder emprego", "inflação alta", "dívida"],
    geo="BR"
)

print(f"Total: {len(eventos_cpi_only)} linhas")
print(f"Maior surpresa: {eventos_cpi_only['surpresa_zscore'].abs().max():.2f}")

Total: 39 linhas
Maior surpresa: 3.02


In [17]:
eventos_cpi_only.to_csv("../data/eventos_completo.csv", index=False)

In [5]:
payroll_bruto = pd.read_csv("../data/payroll.csv", sep=";")
payroll_bruto = payroll_bruto.dropna(subset=["Actual", "Forecast"])

payroll_bruto["Actual"] = payroll_bruto["Actual"].str.replace("K", "", regex=False).astype(float)
payroll_bruto["Forecast"] = payroll_bruto["Forecast"].str.replace("K", "", regex=False).astype(float)

data_limpa = payroll_bruto["Release date"].str.split("(").str[0]
data_limpa = data_limpa.str.replace(r"\s+", " ", regex=True).str.strip()

payroll_bruto["data"] = pd.to_datetime(data_limpa, format="%b %d, %Y", errors="coerce")
print(payroll_bruto[["Release date", "data"]].head(10))
print(f"Quantos viraram NaT: {payroll_bruto['data'].isna().sum()}")

          Release date       data
0   Aug 07, 2026 (Jul) 2026-08-07
1   Jul 02, 2026 (Jun) 2026-07-02
2   Jun 05, 2026 (May) 2026-06-05
3   May 08, 2026 (Apr) 2026-05-08
4   Apr 03, 2026 (Mar) 2026-04-03
5   Mar 06, 2026 (Feb) 2026-03-06
6   Feb 11, 2026 (Jan) 2026-02-11
7   Jan 09, 2026 (Dec) 2026-01-09
8   Dec 16, 2025 (Nov) 2025-12-16
10  Nov 20, 2025 (Sep) 2025-11-20
Quantos viraram NaT: 0


In [6]:
payroll_bruto = payroll_bruto[payroll_bruto["data"] >= "2023-01-01"]

payroll_bruto["indicador"] = "Payroll_EUA"
payroll_eventos = payroll_bruto[["indicador", "data", "Actual", "Forecast"]].rename(
    columns={"Actual": "actual", "Forecast": "forecast"}
)
payroll_eventos["data"] = payroll_eventos["data"].dt.strftime("%Y-%m-%d")

print(payroll_eventos)
print(f"\nTotal de linhas: {len(payroll_eventos)}")

      indicador        data  actual  forecast
0   Payroll_EUA  2026-08-07   -23.0      85.0
1   Payroll_EUA  2026-07-02    57.0     114.0
2   Payroll_EUA  2026-06-05   172.0      85.0
3   Payroll_EUA  2026-05-08   115.0      65.0
4   Payroll_EUA  2026-04-03   178.0      65.0
5   Payroll_EUA  2026-03-06   -92.0      58.0
6   Payroll_EUA  2026-02-11   130.0      66.0
7   Payroll_EUA  2026-01-09    50.0      66.0
8   Payroll_EUA  2025-12-16    64.0      51.0
10  Payroll_EUA  2025-11-20   119.0      53.0
11  Payroll_EUA  2025-09-05    22.0      75.0
12  Payroll_EUA  2025-08-01    73.0     106.0
13  Payroll_EUA  2025-07-03   147.0     111.0
14  Payroll_EUA  2025-06-06   139.0     126.0
15  Payroll_EUA  2025-05-02   177.0     138.0
16  Payroll_EUA  2025-04-04   228.0     137.0
17  Payroll_EUA  2025-03-07   151.0     159.0
18  Payroll_EUA  2025-02-07   143.0     169.0
19  Payroll_EUA  2025-01-10   256.0     164.0
20  Payroll_EUA  2024-12-06   227.0     202.0
21  Payroll_EUA  2024-11-01    12.

In [7]:
eventos_principal = pd.read_csv("../data/eventos.csv")
eventos_atualizado = pd.concat([eventos_principal, payroll_eventos], ignore_index=True)
eventos_atualizado = eventos_atualizado.drop_duplicates()
eventos_atualizado.to_csv("../data/eventos.csv", index=False)

print(eventos_atualizado["indicador"].value_counts())

indicador
Payroll_EUA    43
CPI_EUA        39
IPCA_BR        34
Selic_BR       18
Name: count, dtype: int64


In [9]:
import sys
sys.path.append("..")
from src.features import calcular_surpresa, calcular_ian, calcular_ice

payroll_completo = eventos_atualizado[eventos_atualizado["indicador"] == "Payroll_EUA"].copy()

payroll_completo = calcular_surpresa(payroll_completo)
payroll_completo = calcular_ian(payroll_completo, termos_busca=["nonfarm payrolls", "jobs report"], geo="US")
payroll_completo = calcular_ice(
    payroll_completo,
    termos_otimistas=["abrir empresa", "comprar carro", "promoção passagens"],
    termos_pessimistas=["perder emprego", "inflação alta", "dívida"],
    geo="BR"
)

print(payroll_completo[["data", "surpresa_zscore", "IAN", "ICE"]])

         data  surpresa_zscore       IAN       ICE
0  2023-01-06         0.267704  0.020619  0.464985
1  2023-02-03         3.864244  0.082474  0.432723
2  2023-03-10         1.233765  0.206186  0.390866
3  2023-04-07        -0.034918  0.092784  0.356235
4  2023-05-05         0.849668  0.020619  0.320673
5  2023-06-02         1.850647  0.051546  0.284274
6  2023-07-07        -0.186229  0.113402  0.237769
7  2023-08-04        -0.151311  0.041237  0.199929
8  2023-09-01         0.197868  0.000000  0.161715
9  2023-10-06         1.932122  0.175258  0.113808
10 2023-11-03        -0.349179  0.010309  0.073509
11 2023-12-08         0.221146  0.082474  0.030448
12 2024-01-05         0.535407  0.030928  0.499069
13 2024-02-02         1.932122  0.051546  0.062317
14 2024-03-08         0.896225  0.051546 -0.086693
15 2024-04-05         1.059175  0.113402 -0.096785
16 2024-05-03        -0.733275  0.030928  0.410936
17 2024-06-07         1.047536  0.092784 -0.742297
18 2024-07-05         0.174589 

In [10]:
payroll_completo.to_csv("../data/eventos_payroll_completo.csv", index=False)

In [11]:
import pandas as pd

cpi = pd.read_csv("../data/eventos_completo.csv")
ipca = pd.read_csv("../data/eventos_ipca_completo.csv")
selic = pd.read_csv("../data/eventos_selic_completo.csv")
payroll = pd.read_csv("../data/eventos_payroll_completo.csv")

eventos_todos = pd.concat([cpi, ipca, selic, payroll], ignore_index=True)
eventos_todos = eventos_todos.drop_duplicates(subset=["indicador", "data"], keep="last")
eventos_todos.to_csv("../data/eventos_todos_completo.csv", index=False)
print(eventos_todos["indicador"].value_counts())

eventos_todos.to_csv("../data/eventos_todos_completo.csv", index=False)
print(eventos_todos["indicador"].value_counts())

indicador
Payroll_EUA    43
CPI_EUA        39
IPCA_BR        34
Selic_BR       18
Name: count, dtype: int64
indicador
Payroll_EUA    43
CPI_EUA        39
IPCA_BR        34
Selic_BR       18
Name: count, dtype: int64


In [13]:
eventos_bruto = pd.read_csv("../data/eventos.csv")
cpi_isolado = eventos_bruto[eventos_bruto["indicador"] == "CPI_EUA"]

print(f"Total de linhas rotuladas CPI_EUA: {len(cpi_isolado)}")
print(cpi_isolado[["actual", "forecast"]].describe())
print()
print("Valores fora da faixa esperada (-2 a 2):")
print(cpi_isolado[(cpi_isolado["actual"].abs() > 2) | (cpi_isolado["forecast"].abs() > 2)])

Total de linhas rotuladas CPI_EUA: 39
          actual   forecast
count  39.000000  39.000000
mean    0.258974   0.284615
std     0.222093   0.185725
min    -0.400000  -0.100000
25%     0.200000   0.200000
50%     0.300000   0.300000
75%     0.400000   0.350000
max     0.900000   1.000000

Valores fora da faixa esperada (-2 a 2):
Empty DataFrame
Columns: [indicador, data, actual, forecast, diferenca, surpresa_zscore]
Index: []


In [14]:
cpi_check = pd.read_csv("../data/eventos_completo.csv")
print(f"Total de linhas: {len(cpi_check)}")
print(cpi_check[["data", "actual", "forecast", "diferenca"]].to_string())
print()
print(f"Desvio-padrão calculado: {cpi_check['diferenca'].std():.6f}")

Total de linhas: 134
           data  actual  forecast  diferenca
0    2023-01-06  223.00  200.0000    23.0000
1    2023-02-03  517.00  185.0000   332.0000
2    2023-02-09    0.53    0.5550    -0.0250
3    2023-03-10  311.00  205.0000   106.0000
4    2023-03-10    0.84    0.7800     0.0600
5    2023-03-14    0.40    0.4000     0.0000
6    2023-04-07  236.00  239.0000    -3.0000
7    2023-04-11    0.71    0.7700    -0.0600
8    2023-04-12    0.10    0.2000    -0.1000
9    2023-05-05  253.00  180.0000    73.0000
10   2023-05-10    0.40    0.4000     0.0000
11   2023-05-12    0.61    0.5500     0.0600
12   2023-06-02  339.00  180.0000   159.0000
13   2023-06-07    0.23    0.3700    -0.1400
14   2023-06-13    0.10    0.2000    -0.1000
15   2023-07-07  209.00  225.0000   -16.0000
16   2023-07-11   -0.08   -0.1000     0.0200
17   2023-07-12    0.20    0.3000    -0.1000
18   2023-08-03   13.25   13.5000    -0.2500
19   2023-08-04  187.00  200.0000   -13.0000
20   2023-08-10    0.20    0.2000 